# 02 - Sensor Analysis (smartphone IMU + GNSS)

Investigate the sensor channels available for dead reckoning: accelerometer, gyroscope, magnetometer, gravity and orientation, plus GNSS quality. We use one synchronised trip (`Vw16b`).

In [ ]:
import sys, os
from pathlib import Path
for p in ("..", "."):
    if (Path(p) / "src").is_dir():
        sys.path.insert(0, os.path.abspath(p)); break

In [ ]:
import numpy as np, pandas as pd
from src.data.io_vnbd_loader import IOVNBDDataset
from src.data.smartphone_extractor import SmartphoneExtractor
ds = IOVNBDDataset("data/raw")
ex = SmartphoneExtractor(ds)
d = ex.extract_trip("vw16b").data
print("trip vw16b:", len(d), "samples")

## 1. IMU summary statistics

In [ ]:
imu = [c for c in d.columns if c.startswith(("accel", "gyro", "mag", "gravity"))]
d[imu].describe().T.round(4)

## 2. Acceleration magnitude over time

In [ ]:
import matplotlib.pyplot as plt
a = d[["accel_x", "accel_y", "accel_z"]].to_numpy()
mag = np.linalg.norm(a, axis=1)
t = d["timestamp"] - d["timestamp"].iloc[0]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, mag, lw=0.5)
ax.set(xlabel="time since start (s)", ylabel="|a| (m/s^2)", title="Acceleration magnitude")
plt.show()

## 3. Gyro channels

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for ax, c in zip(axes, ["gyro_x", "gyro_y", "gyro_z"]):
    ax.plot(t, d[c], lw=0.5)
    ax.set_ylabel(c)
axes[-1].set_xlabel("time since start (s)")
plt.show()

## 4. Sensor distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, c in zip(axes, ["accel_x", "gyro_x", "mag_x"]):
    ax.hist(d[c].dropna(), bins=60, density=True)
    ax.set_title(c)
plt.tight_layout()
plt.show()

## 5. GNSS characteristics

In [ ]:
g = d.dropna(subset=["latitude_deg"])
print("GNSS fixes:", len(g))
print("speed_kmh stats:", g["speed_kmh"].describe().round(2).to_dict())
print("accuracy (m) stats:", g["position_accuracy_m"].describe().round(2).to_dict())
print("sats stats:", g["gps_satellites"].describe().round(1).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
tg = g["timestamp"] - d["timestamp"].iloc[0]
ax.plot(tg, g["speed_kmh"], lw=0.8)
ax.set(xlabel="time since start (s)", ylabel="speed (kmh)", title="GNSS speed profile")
plt.show()

## 6. Takeaways

- Acceleration magnitude hovers around gravitational constant (9.8) plus dynamics, so gravity dominates raw accel; gravity channels are available separately.
- Gyro rates are small (rad/s) as expected for road driving.
- GNSS fixes are bursty; the pipeline forward-fills them onto the 10 Hz IMU grid.

## 7. Phase 2 calibration diagnostics

Run the calibration pipeline over a processed trip and inspect the new
estimated/calibrated/aligned columns. Uses `data/calibrated/trip_<id>.parquet`
when present, otherwise the pipeline runs in memory over the processed frame.
Guards so the notebook still opens cleanly when data files are not present.

In [ ]:
from src.data.dataset_manager import DatasetManagerfrom src.calibration.calibration_pipeline import CalibrationConfig, CalibrationPipelinedm = DatasetManager('data/raw', 'data/processed')tid = 'vw16b'if tid not in dm.list_trips():    print('trip vw16b not available yet - run scripts/calibrate_dataset.py first'); tid = Noneif tid is not None:    df = dm.load_calibrated_trip(tid)    added = df.filter(regex='(_cal|_aligned|orient_|gravity_est|linear_accel|category)').columns.tolist()    print('columns:', len(df.columns), '| calibration columns:', len(added))    print('gravity mean magnitude:', round(float(df['gravity_est_magnitude'].mean(skipna=True)), 3), 'm/s^2')    print('heading source mix:', df['orient_heading_source'].value_counts().to_dict())

Estimated gravity magnitude should hover near 9.81 m/s^2. Aligned vs calibrated
traces are identical only under the identity (`TOP_NORTH_SCREEN_UP`) preset;
switch the convention in `configs/calibration_config.yaml` to see a real rotation.

In [ ]:
if tid is not None:    import matplotlib.pyplot as plt    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)    axes[0].plot(df['timestamp'], df['gravity_est_magnitude'], lw=0.8)    axes[0].axhline(9.80665, color='r', ls='--', lw=0.7, alpha=0.7); axes[0].set_ylabel('|g| est (m/s^2)')    axes[1].plot(df['timestamp'], df['orient_yaw_deg'], lw=0.8); axes[1].set_ylabel('yaw (deg)')    axes[2].plot(df['timestamp'], df['accel_x_aligned'], lw=0.8, label='aligned ax');    axes[2].plot(df['timestamp'], df['accel_x_cal'], lw=0.8, alpha=0.5, label='cal ax')    axes[2].legend(); axes[2].set_ylabel('accel (m/s^2)')    plt.xlabel('timestamp (s)'); plt.tight_layout(); plt.show()

## 8. ML-ready window sequences
Load the fixed-length window frames generated over the Phase 1 trip splits
and sanity-check provenance / leakage guarantees.

In [ ]:
from src.data.dataset_manager import DatasetManagerdm2 = DatasetManager('data/raw', 'data/processed')t_frame = Nonetry:    t_frame = dm2.load_split('train')    v_frame = dm2.load_split('validation')    print('train:', t_frame['sequence_id'].nunique(), 'windows of',          sorted(t_frame.groupby('sequence_id').size().unique()), 'rows')    tr, va = set(t_frame['trip_id']), set(v_frame['trip_id'])    print('trip leakage across splits:', bool(tr & va))    print('provenance cols:', [c for c in t_frame.columns          if c.startswith(('sequence_id', 'window_', 'trip_id', 'split', 'sample_in_window'))])except FileNotFoundError as e:    print('sequences not generated yet:', e)